In [1]:
from pathlib import Path
import re
import pandas as pd

BASE_PATH = Path(
    "/Users/manalaltammami/Desktop/twain_child_corpus"
)

text_path = BASE_PATH / "Prince_and_the_Pauper_narrative.txt"
prince_text = text_path.read_text(encoding="utf-8")

# Same word-token rule as the updated corpus analyses.
TOKEN_PATTERN = re.compile(r"[a-z]+(?:['-][a-z]+)*")

def tokenize(text):
    return TOKEN_PATTERN.findall(
        text.lower().replace("’", "'")
    )

total_tokens = len(tokenize(prince_text))

assert total_tokens == 67007, (
    f"Expected 67,007 tokens; found {total_tokens:,}. "
    "Check that you loaded the final narrative file."
)

print(f"The Prince and the Pauper: {total_tokens:,} word tokens")
print(f"Opening quotes: {prince_text.count('“'):,}")
print(f"Closing quotes: {prince_text.count('”'):,}")


The Prince and the Pauper: 67,007 word tokens
Opening quotes: 903
Closing quotes: 893


In [3]:
from datetime import datetime
import hashlib

passages = []
start = None
boundary_note = ""

def add_passage(start, end, note):
    passage = prince_text[start:end]

    if not tokenize(passage):
        return

    passages.append({
        "Passage ID": f"Prince_Q{len(passages) + 1:04d}",
        "Start": start,
        "End": end,
        "Word tokens": len(tokenize(passage)),
        "Before": prince_text[max(0, start - 1000):start],
        "Quoted passage": passage,
        "After": prince_text[end:end + 1000],
        "Boundary note": note,
        "Quotation type": "",
        "Speaker": "",
        "Review notes": "",
    })

for position, character in enumerate(prince_text):
    if character == "“":
        if start is not None:
            add_passage(
                start,
                position,
                "Repeated opening quote: review continuation"
            )
            boundary_note = (
                "Starts after repeated opening quote: "
                "review continuation"
            )
        else:
            boundary_note = ""

        start = position + 1

    elif character == "”":
        if start is None:
            raise ValueError(
                f"Closing quote without opening at {position}."
            )

        add_passage(start, position, boundary_note)
        start = None
        boundary_note = ""

if start is not None:
    raise ValueError("An opening quote remains unclosed at the end.")

prince_passages = pd.DataFrame(passages)

prince_passages["Source SHA256"] = hashlib.sha256(
    prince_text.encode("utf-8")
).hexdigest()

review_folder = BASE_PATH / "prince_register_review"
review_folder.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
review_path = (
    review_folder
    / f"Prince_all_quoted_passages_{timestamp}.csv"
)

prince_passages.to_csv(review_path, index=False)

print(f"Quoted passages: {len(prince_passages):,}")
print(
    "Quoted word tokens:",
    f"{prince_passages['Word tokens'].sum():,}"
)
print(f"Saved to: {review_path}")
print("\nSpeaker labels are blank for manual review.")


Quoted passages: 903
Quoted word tokens: 23,229
Saved to: /Users/manalaltammami/Desktop/twain_child_corpus/prince_register_review/Prince_all_quoted_passages_20261105_003951_730429.csv

Speaker labels are blank for manual review.


In [13]:
from pathlib import Path
import re
import pandas as pd

# Look in the likely download locations
folders = [
    Path.home() / "Downloads",
    Path.home() / "Desktop",
    Path.home() / "Desktop" / "twain_child_corpus"
    / "prince_register_review",
    Path.cwd(),
]

review = None

files = {
    file
    for folder in folders if folder.is_dir()
    for file in folder.glob("*.csv")
    if "prince" in file.name.lower()
}

# Check newest files first; skip blank extraction files
for file in sorted(files, key=lambda p: p.stat().st_mtime, reverse=True):
    candidate = pd.read_csv(file).fillna("")

    required = {"Speaker", "Quotation type", "Speech text"}

    if required.issubset(candidate.columns):
        if candidate["Speaker"].isin(["Tom Canty", "Edward"]).any():
            review = candidate
            print("Loaded:", file)
            break

if review is None:
    print(
        "The reviewed CSV was not found. "
        "Download it using the link below, then rerun this cell."
    )
else:
    def tokenize(text):
        return re.findall(
            r"[a-z]+(?:['-][a-z]+)*",
            text.lower().replace("’", "'")
        )

    speech = review.loc[
        review["Quotation type"].eq("Speech")
        & review["Speaker"].isin(["Tom Canty", "Edward"])
    ].copy()

    speech["Words"] = speech["Speech text"].map(
        lambda text: len(tokenize(text))
    )

    print("\nRetained spoken words:")
    print(speech.groupby("Speaker")["Words"].sum())

Loaded: /Users/manalaltammami/Downloads/Prince_all_quoted_passages_20261101_025412_270072.csv

Retained spoken words:
Speaker
Edward       2929
Tom Canty    2661
Name: Words, dtype: int64


In [15]:
from collections import Counter

pronouns = ["thou", "thy", "thee", "thine", "ye"]
rows = []

for name in ["Tom Canty", "Edward"]:
    passages = speech.loc[speech["Speaker"].eq(name), "Speech text"]

    words = [
        word
        for passage in passages
        for word in tokenize(passage)
    ]

    counts = Counter(words)

    for pronoun in pronouns:
        rows.append({
            "Pronoun": pronoun,
            "Speaker": name,
            "Count": counts[pronoun],
            "Per 10,000": counts[pronoun] / len(words) * 10000
        })

pronoun_results = pd.DataFrame(rows)

print(
    pronoun_results.pivot(
        index="Pronoun",
        columns="Speaker",
        values=["Count", "Per 10,000"]
    ).reindex(pronouns).round(2).to_string()
)

         Count           Per 10,000          
Speaker Edward Tom Canty     Edward Tom Canty
Pronoun                                      
thou      56.0      29.0     191.19    108.98
thy       35.0      31.0     119.49    116.50
thee      33.0      18.0     112.67     67.64
thine      3.0       4.0      10.24     15.03
ye        10.0       6.0      34.14     22.55


In [17]:
markers = [
    "methinks", "peradventure", "tis",
    "grace", "lordship", "varlet",
    "art", "hast", "wilt"
]

rows = []

for name in ["Tom Canty", "Edward"]:
    passages = speech.loc[speech["Speaker"].eq(name), "Speech text"]

    words = [
        word
        for passage in passages
        for word in tokenize(passage)
    ]

    counts = Counter(words)

    for marker in markers:
        rows.append({
            "Form": marker,
            "Speaker": name,
            "Count": counts[marker],
            "Per 10,000": counts[marker] / len(words) * 10000
        })

marker_results = pd.DataFrame(rows)

print(
    marker_results.pivot(
        index="Form",
        columns="Speaker",
        values=["Count", "Per 10,000"]
    ).reindex(markers).round(2).to_string()
)

              Count           Per 10,000          
Speaker      Edward Tom Canty     Edward Tom Canty
Form                                              
methinks        1.0       2.0       3.41      7.52
peradventure    0.0       1.0       0.00      3.76
tis             9.0      10.0      30.73     37.58
grace           2.0       3.0       6.83     11.27
lordship        0.0       1.0       0.00      3.76
varlet          0.0       0.0       0.00      0.00
art             9.0       5.0      30.73     18.79
hast           13.0       2.0      44.38      7.52
wilt            3.0       0.0      10.24      0.00
